# Model C: Time-Series + Transformer (FinBERT dan DistilRoBERTa)

Model C memakai baris dan split yang sama dengan Model A dan B. Teks berita diambil
dari `data/export/articles.parquet`, lalu embedding judul diagregasi per
`aligned_trading_date` (hari bursa versi WIB) dengan mean pooling dan L2 normalization.
Hari tanpa berita diisi vektor nol.

| Varian | Fusion | Fitur |
|---|---|---|
| `A` | - | 7 fitur kurs |
| `C_finbert_early` | Early | 7 kurs + 64 komponen PCA FinBERT |
| `C_roberta_early` | Early | 7 kurs + 64 komponen PCA DistilRoBERTa |
| `C_combined` | Early | 7 kurs + 64 + 64 komponen PCA |
| `C_*_late` | Late | alpha x prediksi kurs + (1-alpha) x prediksi teks, alpha dari Val |

PCA dan scaler hanya di-fit di train.

## 0. Import dan lokasi file

In [1]:
import json
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch
from transformers import AutoTokenizer, AutoModel

import sklearn, xgboost, lightgbm
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, BayesianRidge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor, ExtraTreesRegressor,
    AdaBoostRegressor, GradientBoostingRegressor,
)
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.inspection import permutation_importance
from sklearn.manifold import TSNE
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

SEED = 42
N_PCA = 64
EMB_MODELS = {"finbert": "ProsusAI/finbert", "roberta": "distilroberta-base"}
warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda x: f"{x:.6g}")
torch.set_num_threads(max(1, torch.get_num_threads()))


def find_root(start):
    # Notebook bisa dijalankan dari folder mana pun, jadi root dicari ke atas.
    for cand in [Path(start).resolve(), *Path(start).resolve().parents]:
        if (cand / "data" / "export" / "prices_daily.parquet").exists():
            return cand
    raise FileNotFoundError("Folder proyek (berisi data/export/prices_daily.parquet) tidak ditemukan")


ROOT = find_root(Path.cwd())
EXPORT_DIR = ROOT / "data" / "export"
OUT_DIR = ROOT / "trial-and-error" / "Ken" / "results"
sys.path.insert(0, str((ROOT / "src").resolve()))
from eval_utils import evaluate, METRICS

print("Proyek :", ROOT)
print("Metrik :", METRICS, "| torch", torch.__version__, "| cuda", torch.cuda.is_available())

Proyek : D:\nlpproject-grestercinta
Metrik : ['MAE', 'MSE', 'RMSE', 'DirAcc'] | torch 2.14.1+cpu | cuda False


## 1. Kurs dan split

In [2]:
prices = pd.read_parquet(EXPORT_DIR / "prices_daily.parquet")
prices["date"] = pd.to_datetime(prices["date"]).dt.normalize()
prices = prices.sort_values("date").reset_index(drop=True)

train = prices[prices["split"] == "train"].reset_index(drop=True)
val = prices[prices["split"] == "val"].reset_index(drop=True)
test = prices[prices["split"] == "test"].reset_index(drop=True)

for name, part in [("Train", train), ("Val", val), ("Test", test)]:
    print(f"{name:5}: {len(part):4d} | {part['date'].min().date()} -> {part['date'].max().date()}")

FEAT_A = ["kurs_lag1", "kurs_lag2", "kurs_lag3", "kurs_lag4", "kurs_lag5",
          "kurs_roll_mean5", "kurs_roll_std5"]
ytr, yva, yte = train["target"].values, val["target"].values, test["target"].values

a = pd.read_parquet(OUT_DIR / "dataset_model_A.parquet")
a["date"] = pd.to_datetime(a["Tanggal_dt"]).dt.normalize()
cea = prices[["date", "target"]].merge(a[["date", "target"]], on="date", suffixes=("", "_A"))
print("tanggal cocok dengan Model A:", len(cea), "dari", len(prices))
print("selisih target maks:", float((cea["target"] - cea["target_A"]).abs().max()))

Train:  836 | 2021-09-08 -> 2025-02-18
Val  :  178 | 2025-02-20 -> 2025-11-20
Test :  180 | 2025-11-24 -> 2026-08-31
tanggal cocok dengan Model A: 1194 dari 1194
selisih target maks: 0.0


## 2. Artikel dan embedding harian

In [3]:
articles = pd.read_parquet(EXPORT_DIR / "articles.parquet")
articles["tanggal_bursa"] = pd.to_datetime(articles["aligned_trading_date"]).dt.normalize()
articles = articles.sort_values(["tanggal_bursa", "published_wib"]).reset_index(drop=True)

in_calendar = articles["tanggal_bursa"].isin(prices["date"])
print("artikel total          :", len(articles))
print("di luar kalender bursa :", int((~in_calendar).sum()))
print("hari bursa berita      :", articles.loc[in_calendar, "tanggal_bursa"].nunique())
print("berita per hari        : median", float(articles.groupby("tanggal_bursa").size().median()))


def encode_titles(texts, model_name, batch_size=32, max_length=64):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).eval()
    chunks = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        enc = tokenizer(batch, padding=True, truncation=True, max_length=max_length,
                        return_tensors="pt")
        with torch.no_grad():
            out = model(**enc).last_hidden_state[:, 0]
        chunks.append(torch.nn.functional.normalize(out, dim=1).numpy())
        if start % 1024 == 0:
            print("  ", start, "/", len(texts))
    return np.vstack(chunks).astype(np.float32)


def daily_embeddings(tag, model_name):
    cache = OUT_DIR / f"embeddings_daily_{tag}.parquet"
    meta_path = OUT_DIR / f"embeddings_daily_{tag}.json"
    if cache.exists() and meta_path.exists():
        meta = json.loads(meta_path.read_text(encoding="utf-8"))
        if meta.get("model") == model_name and meta.get("n_artikel") == len(articles):
            print(tag, ": memakai cache", cache.name)
            return pd.read_parquet(cache)

    vec = encode_titles(articles["title"].str.strip().tolist(), model_name)
    dates = articles["tanggal_bursa"].values
    rows = [vec[dates == d].mean(axis=0) for d in pd.unique(articles["tanggal_bursa"])]
    daily = pd.DataFrame(np.vstack(rows), index=pd.unique(articles["tanggal_bursa"]))
    daily = daily.div(np.linalg.norm(daily.values, axis=1, keepdims=True), axis=0)
    daily.index.name = "date"
    daily.to_parquet(cache)
    meta_path.write_text(json.dumps({"model": model_name, "n_artikel": int(len(articles)),
                                     "dim": int(daily.shape[1])}, indent=2), encoding="utf-8")
    print(tag, ": tersimpan", cache.name, daily.shape)
    return daily


DAILY = {tag: daily_embeddings(tag, name) for tag, name in EMB_MODELS.items()}
for tag, e in DAILY.items():
    print(f"{tag:8}: {e.shape[0]} hari, dim {e.shape[1]}")

artikel total          : 4332
di luar kalender bursa : 4
hari bursa berita      : 1073
berita per hari        : median 3.0
   0 / 4332
   1024 / 4332
   2048 / 4332
   3072 / 4332
   4096 / 4332
finbert : tersimpan embeddings_daily_finbert.parquet (1076, 768)
   0 / 4332
   1024 / 4332
   2048 / 4332
   3072 / 4332
   4096 / 4332
roberta : tersimpan embeddings_daily_roberta.parquet (1076, 768)
finbert : 1076 hari, dim 768
roberta : 1076 hari, dim 768


## 3. Early fusion: kurs + PCA embedding

In [4]:
def embedding_matrix(part, tag):
    M = DAILY[tag].reindex(part["date"]).values.astype(float)
    miss = np.isnan(M).any(axis=1)
    if miss.any():
        print(f"  {tag}: {int(miss.sum())} hari tanpa berita diisi 0")
    return np.nan_to_num(M, nan=0.0)


emb_scaler = {tag: StandardScaler().fit(embedding_matrix(train, tag)) for tag in DAILY}
RAW = {(tag, s): emb_scaler[tag].transform(embedding_matrix(p, tag))
       for tag in DAILY for s, p in [("train", train), ("val", val), ("test", test)]}

PCA = {}
for tag in DAILY:
    svd = TruncatedSVD(n_components=N_PCA, random_state=SEED)
    PCA[(tag, "train")] = svd.fit_transform(RAW[(tag, "train")])
    for s in ("val", "test"):
        PCA[(tag, s)] = svd.transform(RAW[(tag, s)])
    print(f"{tag:8}: PCA {N_PCA} komponen, explained={svd.explained_variance_ratio_.sum():.3f}")

ts_scaler = StandardScaler().fit(train[FEAT_A])
TS = {s: ts_scaler.transform(p[FEAT_A]) for s, p in [("train", train), ("val", val), ("test", test)]}

EF = {}
for tag in DAILY:
    for s in ("train", "val", "test"):
        EF[(tag, s)] = np.hstack([TS[s], PCA[(tag, s)]])
for s in ("train", "val", "test"):
    EF[("combined", s)] = np.hstack([EF[("finbert", s)], PCA[("roberta", s)]])

for tag in ("finbert", "roberta", "combined"):
    print(f"{tag:8}: {EF[(tag, 'train')].shape[1]} fitur")

  finbert: 102 hari tanpa berita diisi 0
  roberta: 102 hari tanpa berita diisi 0
  finbert: 102 hari tanpa berita diisi 0
  finbert: 15 hari tanpa berita diisi 0
  finbert: 4 hari tanpa berita diisi 0
  roberta: 102 hari tanpa berita diisi 0
  roberta: 15 hari tanpa berita diisi 0
  roberta: 4 hari tanpa berita diisi 0
finbert : PCA 64 komponen, explained=0.847
roberta : PCA 64 komponen, explained=0.886
finbert : 71 fitur
roberta : 71 fitur
combined: 135 fitur


## 4. Model, evaluasi, dan pelatihan

In [5]:
MODELS = {
    "LinearRegression": lambda: LinearRegression(),
    "Ridge":            lambda: Ridge(),
    "Lasso":            lambda: Lasso(),
    "ElasticNet":       lambda: ElasticNet(),
    "BayesianRidge":    lambda: BayesianRidge(),
    "KNN":              lambda: KNeighborsRegressor(),
    "SVR":              lambda: SVR(),
    "DecisionTree":     lambda: DecisionTreeRegressor(random_state=SEED),
    "RandomForest":     lambda: RandomForestRegressor(random_state=SEED, n_jobs=-1),
    "ExtraTrees":       lambda: ExtraTreesRegressor(random_state=SEED, n_jobs=-1),
    "AdaBoost":         lambda: AdaBoostRegressor(random_state=SEED),
    "GradientBoosting": lambda: GradientBoostingRegressor(random_state=SEED),
    "XGBoost":          lambda: XGBRegressor(random_state=SEED, n_jobs=-1, verbosity=0),
    "LightGBM":         lambda: LGBMRegressor(random_state=SEED, n_jobs=-1, verbosity=-1),
}

VARIANTS = {
    "A": TS,
    "C_finbert_early": {s: EF[("finbert", s)] for s in ("train", "val", "test")},
    "C_roberta_early": {s: EF[("roberta", s)] for s in ("train", "val", "test")},
    "C_combined": {s: EF[("combined", s)] for s in ("train", "val", "test")},
}

rows, preds = [], []


def add_row(varian, model_name, split, part, y, y_pred, n_fitur):
    rows.append({"Varian": varian, "Model": model_name, "Split": split,
                 "n_fitur": n_fitur, **evaluate(y, y_pred)})
    preds.append(pd.DataFrame({"Varian": varian, "Model": model_name, "Split": split,
                               "date": part["date"].values, "y_true": y, "y_pred": y_pred}))


for varian, X in VARIANTS.items():
    n_fitur = X["train"].shape[1]
    for model_name, make_model in MODELS.items():
        model = make_model().fit(X["train"], ytr)
        for split, part, y in [("Val", val, yva), ("Test", test, yte)]:
            add_row(varian, model_name, split, part, y, model.predict(X[split.lower()]), n_fitur)
    print("selesai:", varian, "|", n_fitur, "fitur")

results = pd.DataFrame(rows)
results["Degenerate"] = results["PredStd"] < 1e-9
print("total baris:", len(results), "| degenerate:", int(results["Degenerate"].sum()))

selesai: A | 7 fitur
selesai: C_finbert_early | 71 fitur
selesai: C_roberta_early | 71 fitur
selesai: C_combined | 135 fitur
total baris: 112 | degenerate: 24


## 5. Late fusion

In [6]:
ts_model = BayesianRidge().fit(TS["train"], ytr)
pred_ts = {s: ts_model.predict(TS[s]) for s in ("train", "val", "test")}

LATE = {}
for tag in ("finbert", "roberta", "combined"):
    if tag == "combined":
        Xe = {s: np.hstack([RAW[("finbert", s)], RAW[("roberta", s)]])
              for s in ("train", "val", "test")}
    else:
        Xe = {s: RAW[(tag, s)] for s in ("train", "val", "test")}

    text_model = Ridge(alpha=100.0).fit(Xe["train"], ytr)
    pred_txt = {s: text_model.predict(Xe[s]) for s in ("train", "val", "test")}

    best_alpha, best_rmse = 0.5, np.inf
    for alpha in np.linspace(0, 1, 101):
        rmse = np.sqrt(np.mean((alpha * pred_ts["val"] + (1 - alpha) * pred_txt["val"] - yva) ** 2))
        if rmse < best_rmse:
            best_rmse, best_alpha = rmse, alpha

    varian = f"C_{tag}_late"
    n_fitur = len(FEAT_A) + Xe["train"].shape[1]
    LATE[tag] = {"alpha": best_alpha, "pred": pred_txt, "n_fitur": n_fitur}
    for split, part, y, key in [("Val", val, yva, "val"), ("Test", test, yte, "test")]:
        yp = best_alpha * pred_ts[key] + (1 - best_alpha) * pred_txt[key]
        add_row(varian, f"Ridge_alpha{best_alpha:.2f}", split, part, y, yp, n_fitur)
    print(f"{varian:16}: alpha_ts={best_alpha:.2f}, Val RMSE={best_rmse:.6f}")

results = pd.DataFrame(rows)
results["Degenerate"] = results["PredStd"] < 1e-9
print("total baris setelah late fusion:", len(results))

C_finbert_late  : alpha_ts=0.83, Val RMSE=0.003373
C_roberta_late  : alpha_ts=1.00, Val RMSE=0.003389
C_combined_late : alpha_ts=0.97, Val RMSE=0.003388
total baris setelah late fusion: 118


## 6. Hasil per varian

In [7]:
for split in ("Val", "Test"):
    t = (results[(results["Split"] == split) & (~results["Degenerate"])]
         .sort_values(["Varian", "RMSE"])
         .groupby("Varian").head(3)
         .set_index(["Varian", "Model"])[["n_fitur", "RMSE", "MAE", "DirAcc", "PredStd"]])
    print(f"\n=== {split} | 3 model terbaik per varian ===")
    print(t.to_string())


=== Val | 3 model terbaik per varian ===
                                  n_fitur       RMSE        MAE   DirAcc     PredStd
Varian          Model                                                               
A               BayesianRidge           7 0.00338869 0.00250102 0.448864  0.00025135
                Ridge                   7 0.00339263 0.00251092 0.460227 0.000353976
                LinearRegression        7 0.00340153 0.00252672 0.465909 0.000473432
C_combined      RandomForest          135 0.00346523 0.00258257 0.477273 0.000716911
                GradientBoosting      135 0.00347231 0.00265876 0.477273  0.00106372
                ExtraTrees            135   0.003474 0.00258458 0.556818 0.000878208
C_combined_late Ridge_alpha0.97      1543 0.00338795 0.00250512 0.448864 0.000255795
C_finbert_early BayesianRidge          71 0.00346571 0.00263871 0.465909 0.000866808
                Ridge                  71 0.00348544 0.00270091 0.471591  0.00100057
                LinearR

## 7. Delta RMSE terhadap Model A

In [8]:
base = (results[(results["Varian"] == "A") & (results["Split"] == "Test")]
        .set_index("Model")["RMSE"])
best_a = results[(results["Varian"] == "A") & (results["Split"] == "Test")
                 & (~results["Degenerate"])].sort_values("RMSE").iloc[0]
print(f"Anchor A (Test): {best_a['Model']} RMSE={best_a['RMSE']:.7f} DirAcc={best_a['DirAcc']:.4f}")

delta_rows = []
for var in ("C_finbert_early", "C_roberta_early", "C_combined",
            "C_finbert_late", "C_roberta_late", "C_combined_late"):
    sub = results[(results["Varian"] == var) & (results["Split"] == "Test") & (~results["Degenerate"])]
    best = sub.sort_values("RMSE").iloc[0]
    delta_rows.append({"Varian": var, "Model": best["Model"], "n_fitur": int(best["n_fitur"]),
                       "RMSE": best["RMSE"], "DirAcc": best["DirAcc"],
                       "dRMSE_pct": 100 * (best["RMSE"] - best_a["RMSE"]) / best_a["RMSE"]})
print("\n=== Delta C vs A (Test) ===")
print(pd.DataFrame(delta_rows).to_string(index=False, float_format=lambda x: f"{x:.6g}"))

Anchor A (Test): KNN RMSE=0.0029806 DirAcc=0.5284

=== Delta C vs A (Test) ===
         Varian           Model  n_fitur       RMSE   DirAcc  dRMSE_pct
C_finbert_early    RandomForest       71 0.00291387 0.545455   -2.24012
C_roberta_early    RandomForest       71 0.00299933 0.477273   0.627136
     C_combined    RandomForest      135 0.00283486 0.590909   -4.89071
 C_finbert_late Ridge_alpha0.83      775 0.00298917 0.454545   0.286425
 C_roberta_late Ridge_alpha1.00      775 0.00298408 0.414773    0.11569
C_combined_late Ridge_alpha0.97     1543 0.00298141 0.426136  0.0260884


## 8. Hyperparameter tuning

In [9]:
GRIDS = {
    "Ridge": {"alpha": [0.1, 1.0, 10.0, 100.0]},
    "Lasso": {"alpha": [1e-4, 1e-3, 1e-2]},
    "ElasticNet": {"alpha": [1e-4, 1e-3], "l1_ratio": [0.2, 0.5, 0.8]},
    "KNN": {"n_neighbors": [3, 5, 7, 11, 21], "weights": ["uniform", "distance"]},
    "SVR": {"C": [0.1, 1.0, 10.0], "epsilon": [1e-4, 1e-3]},
    "RandomForest": {"max_depth": [3, 5, None], "min_samples_leaf": [1, 5]},
    "XGBoost": {"max_depth": [2, 3], "learning_rate": [0.03, 0.05], "n_estimators": [300]},
    "LightGBM": {"num_leaves": [7, 15], "learning_rate": [0.03, 0.05], "n_estimators": [300]},
}
CV = TimeSeriesSplit(n_splits=5)

tuned = {}
tune_rows = []
for var in ("C_finbert_early", "C_roberta_early", "C_combined"):
    sub = results[(results["Varian"] == var) & (results["Split"] == "Val") & (~results["Degenerate"])]
    model_name = sub.sort_values("RMSE").iloc[0]["Model"]
    if model_name not in GRIDS:
        model_name = "Ridge"

    X = VARIANTS[var]
    gs = GridSearchCV(MODELS[model_name](), GRIDS[model_name], cv=CV, n_jobs=-1,
                      scoring="neg_root_mean_squared_error")
    gs.fit(X["train"], ytr)
    best = gs.best_estimator_
    tuned[var] = {"model": best, "X": X}

    for split, part, y in [("Val", val, yva), ("Test", test, yte)]:
        yp = best.predict(X[split.lower()])
        tune_rows.append({"Varian": var + "_tuned", "Model": model_name + "+tuned",
                          "Split": split, "n_fitur": X["train"].shape[1],
                          "params": json.dumps(gs.best_params_),
                          "cv_RMSE": float(-gs.best_score_), **evaluate(y, yp)})
        add_row(var + "_tuned", model_name + "+tuned", split, part, y, yp, X["train"].shape[1])
    print(f"{var:18}: {model_name} {gs.best_params_} | cv RMSE {-gs.best_score_:.6f}")

tuning = pd.DataFrame(tune_rows)
tuning["Degenerate"] = tuning["PredStd"] < 1e-9
print(tuning[tuning["Split"] == "Test"][["Varian", "Model", "params", "cv_RMSE", "RMSE", "DirAcc"]]
      .to_string(index=False, float_format=lambda x: f"{x:.6g}"))

C_finbert_early   : Ridge {'alpha': 100.0} | cv RMSE 0.003871
C_roberta_early   : RandomForest {'max_depth': 3, 'min_samples_leaf': 5} | cv RMSE 0.003777
C_combined        : RandomForest {'max_depth': 3, 'min_samples_leaf': 5} | cv RMSE 0.003714
               Varian              Model                                  params    cv_RMSE       RMSE   DirAcc
C_finbert_early_tuned        Ridge+tuned                        {"alpha": 100.0} 0.00387051 0.00302026 0.454545
C_roberta_early_tuned RandomForest+tuned {"max_depth": 3, "min_samples_leaf": 5} 0.00377661 0.00290177 0.551136
     C_combined_tuned RandomForest+tuned {"max_depth": 3, "min_samples_leaf": 5} 0.00371389 0.00288373 0.568182


## 9. Permutation importance

In [10]:
def feature_names(var):
    if var == "C_finbert_early":
        return FEAT_A + [f"finbert_pc{i}" for i in range(1, N_PCA + 1)]
    if var == "C_roberta_early":
        return FEAT_A + [f"roberta_pc{i}" for i in range(1, N_PCA + 1)]
    return (FEAT_A + [f"finbert_pc{i}" for i in range(1, N_PCA + 1)]
            + [f"roberta_pc{i}" for i in range(1, N_PCA + 1)])


imp_rows = []
for var, item in tuned.items():
    Xva = item["X"]["val"]
    r = permutation_importance(item["model"], Xva, yva, n_repeats=20, random_state=SEED,
                               scoring="neg_root_mean_squared_error")
    for feat, mean, std in sorted(zip(feature_names(var), r.importances_mean, r.importances_std),
                                  key=lambda t: -t[1]):
        imp_rows.append({"Varian": var, "Fitur": feat, "importance": mean, "std": std})

imp = pd.DataFrame(imp_rows)
for var in tuned:
    print(f"\n=== {var}: 8 fitur paling berpengaruh (validasi) ===")
    print(imp[imp["Varian"] == var].head(8).to_string(index=False, float_format=lambda x: f"{x:.6g}"))


=== C_finbert_early: 8 fitur paling berpengaruh (validasi) ===
         Varian        Fitur  importance         std
C_finbert_early finbert_pc47 3.02895e-05 1.84944e-05
C_finbert_early finbert_pc12  2.1202e-05 9.54122e-06
C_finbert_early  finbert_pc2 1.83628e-05 8.68797e-06
C_finbert_early finbert_pc60 1.62445e-05 1.10908e-05
C_finbert_early finbert_pc32 1.47737e-05 6.20183e-06
C_finbert_early finbert_pc13 1.21575e-05   1.791e-05
C_finbert_early finbert_pc63 1.12897e-05 4.84751e-06
C_finbert_early  finbert_pc4  7.2032e-06 7.29128e-06

=== C_roberta_early: 8 fitur paling berpengaruh (validasi) ===
         Varian          Fitur  importance         std
C_roberta_early   roberta_pc35 3.07647e-05  7.5267e-06
C_roberta_early kurs_roll_std5 9.15391e-06  5.2338e-06
C_roberta_early   roberta_pc40 5.61141e-06 3.48004e-06
C_roberta_early   roberta_pc56 3.56389e-06 3.83045e-06
C_roberta_early   roberta_pc13 3.25131e-06 3.33468e-06
C_roberta_early    roberta_pc9 2.71181e-06 1.56943e-06
C_roberta_

## 10. t-SNE embedding harian dan volatilitas

In [11]:
vol = prices[["date", "ret"]].copy()
vol["abs_ret"] = vol["ret"].abs()
q_hi = vol["abs_ret"].quantile(0.85)
q_lo = vol["abs_ret"].quantile(0.15)

emb_dates = DAILY["finbert"].index.intersection(vol["date"])
X_emb = DAILY["finbert"].loc[emb_dates].values
vol_map = vol.set_index("date")["abs_ret"].reindex(emb_dates)
label = np.where(vol_map >= q_hi, "High Vol", np.where(vol_map <= q_lo, "Low Vol", None))
mask = label != None

X_plot = X_emb[mask]
labels = label[mask]
print("High Vol:", int((labels == "High Vol").sum()), "| Low Vol:", int((labels == "Low Vol").sum()))

if len(X_plot) >= 20:
    tsne = TSNE(n_components=2, random_state=SEED, perplexity=min(30, len(X_plot) // 3),
                max_iter=1000, init="pca", learning_rate="auto")
    X_2d = tsne.fit_transform(X_plot)
    fig, ax = plt.subplots(figsize=(8, 6))
    for grp, color, marker in [("High Vol", "#b3231f", "o"), ("Low Vol", "#1f4e79", "s")]:
        m = labels == grp
        ax.scatter(X_2d[m, 0], X_2d[m, 1], c=color, marker=marker, s=28, alpha=0.75, label=grp)
    ax.set_title("t-SNE embedding judul FinBERT\nVolatilitas harian USD/IDR: 15% teratas vs 15% terbawah")
    ax.set_xlabel("dim 1")
    ax.set_ylabel("dim 2")
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(OUT_DIR / "tsne_finbert.png", dpi=120, bbox_inches="tight")
    print("Disimpan:", OUT_DIR / "tsne_finbert.png")

High Vol: 164 | Low Vol: 165
Disimpan: D:\nlpproject-grestercinta\trial-and-error\Ken\results\tsne_finbert.png


## 11. Analisis error per tanggal

In [12]:
best_c = results[(results["Varian"].str.startswith("C_")) & (results["Split"] == "Test")
                 & (~results["Degenerate"])].sort_values("RMSE").iloc[0]
pred_table = pd.concat(preds, ignore_index=True)

pa = pred_table[(pred_table["Varian"] == "A") & (pred_table["Model"] == best_a["Model"])
                & (pred_table["Split"] == "Test")].set_index("date")["y_pred"]
pc = pred_table[(pred_table["Varian"] == best_c["Varian"]) & (pred_table["Model"] == best_c["Model"])
                & (pred_table["Split"] == "Test")].set_index("date")["y_pred"]

err = pd.DataFrame({"y_true": yte, "pred_A": pa.values, "pred_C": pc.values}, index=test["date"].values)
err["err_A"] = (err["pred_A"] - err["y_true"]).abs()
err["err_C"] = (err["pred_C"] - err["y_true"]).abs()
nonzero = err["y_true"] != 0
improved = nonzero & (np.sign(err["pred_A"]) != np.sign(err["y_true"])) & (np.sign(err["pred_C"]) == np.sign(err["y_true"]))
worsened = nonzero & (np.sign(err["pred_A"]) == np.sign(err["y_true"])) & (np.sign(err["pred_C"]) != np.sign(err["y_true"]))
err["arah_membaik"] = improved
err["arah_memburuk"] = worsened

print(f"C terbaik (Test): {best_c['Varian']} / {best_c['Model']} RMSE={best_c['RMSE']:.7f}")
print("arah membaik:", int(improved.sum()), "hari | arah memburuk:", int(worsened.sum()), "hari",
      "| hari test:", int(nonzero.sum()))
err.to_csv(OUT_DIR / "error_analysis_model_C.csv")
print("\n5 hari dengan perbaikan arah terbesar:")
print(err[improved].nlargest(5, "err_A")[["y_true", "pred_A", "pred_C"]].to_string())

C terbaik (Test): C_combined / RandomForest RMSE=0.0028349
arah membaik: 47 hari | arah memburuk: 36 hari | hari test: 176

5 hari dengan perbaikan arah terbesar:
                y_true       pred_A       pred_C
2026-05-18   0.0096696 -0.000927105  0.000477749
2026-06-04  0.00600502 -0.000927105  0.000870038
2026-08-10 -0.00660919  6.10445e-05 -0.000379544
2026-03-10 -0.00561252  6.10445e-05 -0.000462479
2026-02-06  0.00361879  -0.00136867  0.000470456


## 12. Ekspor dan ringkasan

In [13]:
results["params"] = ""
results["cv_RMSE"] = np.nan
all_metrics = pd.concat([results, tuning], ignore_index=True)
all_metrics.to_csv(OUT_DIR / "metrics_model_C.csv", index=False)

pred_df = pd.concat(preds, ignore_index=True)
pred_df[pred_df["Split"] == "Val"].to_csv(OUT_DIR / "predictions_val_model_C.csv", index=False)
pred_df[pred_df["Split"] == "Test"].to_csv(OUT_DIR / "predictions_test_model_C.csv", index=False)
tuning.to_csv(OUT_DIR / "tuning_model_C.csv", index=False)
imp.to_csv(OUT_DIR / "importance_model_C.csv", index=False)

meta = {
    "model": "C",
    "seed": SEED,
    "sumber": ["data/export/prices_daily.parquet", "data/export/articles.parquet"],
    "n_baris": int(len(prices)),
    "n_artikel": int(len(articles)),
    "split": {k: int((prices["split"] == k).sum()) for k in ("train", "val", "test")},
    "embedding": {"model": EMB_MODELS, "pooling": "mean harian + L2", "n_pca": N_PCA,
                  "alpha_late": {k: v["alpha"] for k, v in LATE.items()}},
    "versi": {"torch": torch.__version__, "transformers": __import__("transformers").__version__,
              "sklearn": sklearn.__version__, "xgboost": xgboost.__version__,
              "lightgbm": lightgbm.__version__},
}
(OUT_DIR / "metadata_model_C.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")

summary = []
for var in ("A", "C_finbert_early", "C_roberta_early", "C_combined",
            "C_finbert_late", "C_roberta_late", "C_combined_late"):
    sub = all_metrics[(all_metrics["Varian"] == var) & (all_metrics["Split"] == "Test")
                      & (~all_metrics["Degenerate"])]
    best = sub.sort_values("RMSE").iloc[0]
    summary.append({"Varian": var, "Model": best["Model"], "n_fitur": int(best["n_fitur"]),
                    "RMSE": best["RMSE"], "MAE": best["MAE"], "DirAcc": best["DirAcc"]})
print("=== Model terbaik per varian (Test) ===")
print(pd.DataFrame(summary).to_string(index=False, float_format=lambda x: f"{x:.6g}"))

=== Model terbaik per varian (Test) ===
         Varian           Model  n_fitur       RMSE        MAE   DirAcc
              A             KNN        7 0.00298064 0.00231492 0.528409
C_finbert_early    RandomForest       71 0.00291387 0.00222471 0.545455
C_roberta_early    RandomForest       71 0.00299933 0.00231481 0.477273
     C_combined    RandomForest      135 0.00283486 0.00214303 0.590909
 C_finbert_late Ridge_alpha0.83      775 0.00298917  0.0022884 0.454545
 C_roberta_late Ridge_alpha1.00      775 0.00298408 0.00229396 0.414773
C_combined_late Ridge_alpha0.97     1543 0.00298141 0.00229021 0.426136
